In [ ]:
import os
import traceback
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException # type: ignore

spark = SparkSession.builder \
    .master("spark://192.168.1.13:7077") \
    .appName("chembl_eda") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://192.168.1.13:5433/chembl_36"
properties = {
    "user": "chembl",
    "password": "chembl",
    "driver": "org.postgresql.Driver"
}

try:
    df = spark.read.jdbc(
        url=jdbc_url,
        table="action_type",
        properties=properties
    )
    
    print("✅ DataFrame schema:")
    df.printSchema()
    
    print("✅ First 5 rows:")
    print(df)
    
    df.createOrReplaceTempView("action_type_view")
    
    sql_df = spark.sql("SELECT * FROM action_type_view LIMIT 5")
    print("✅ SQL query result:")
    sql_df.show()

except AnalysisException as ae:
    print("❌ Table does not exist yet:", ae)
except Exception as e:
    print("❌ Other error:")
    traceback.print_exc()
